In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/vangogh_restore

/content/drive/MyDrive/vangogh_restore


In [ ]:
!python step1_build_dataset.py --input_dir ./vangogh_color

Tìm thấy 188 ảnh gốc.
  Train: 150 | Test: 38

Hoàn thành! Đã tạo 188 cặp ảnh, bỏ qua 0 ảnh lỗi.
Dataset lưu tại: ./dataset_restore/
Xem 6 ảnh mẫu tại: ./dataset_restore/samples/


In [ ]:
!rm -rf dataset_restore
!rm -rf output_pix2pix

In [ ]:
!python step2_train_pix2pix.py --epochs 5

Device: cuda
Train: 150 ảnh | Test: 38 ảnh

Bắt đầu train 5 epochs...
Epoch    1/5 | D:0.7802 G:0.8758 L1:0.2765 | 6.4s
Epoch    2/5 | D:0.5895 G:1.0160 L1:0.2097 | 5.6s
Epoch    3/5 | D:0.5212 G:1.3591 L1:0.2004 | 5.7s
Epoch    4/5 | D:0.4864 G:1.4706 L1:0.1976 | 5.8s
Epoch    5/5 | D:0.4453 G:1.6043 L1:0.1940 | 5.7s

Train xong! Checkpoint tốt nhất lưu tại: ./output_pix2pix/checkpoints/generator_best.pth
Ảnh mẫu lưu tại: ./output_pix2pix/samples/


In [ ]:
!python step2_train_pix2pix.py --epochs 100

Device: cuda
Train: 150 ảnh | Test: 38 ảnh

Bắt đầu train 100 epochs...
Epoch    1/100 | D:0.7772 G:0.8944 L1:0.2681 | 6.2s
Epoch    2/100 | D:0.5891 G:1.0870 L1:0.2066 | 5.7s
Epoch    3/100 | D:0.4851 G:1.4949 L1:0.2003 | 5.7s
Epoch    4/100 | D:0.4925 G:1.4572 L1:0.1996 | 5.9s
Epoch    5/100 | D:0.5052 G:1.5723 L1:0.1982 | 5.8s
Epoch    6/100 | D:0.4756 G:1.5902 L1:0.1907 | 5.9s
Epoch    7/100 | D:0.5425 G:1.4165 L1:0.1865 | 5.8s
Epoch    8/100 | D:0.5819 G:1.2693 L1:0.1801 | 5.9s
Epoch    9/100 | D:0.5338 G:1.2670 L1:0.1721 | 6.0s
Epoch   10/100 | D:0.5674 G:1.3110 L1:0.1705 | 6.0s
Epoch   11/100 | D:0.5197 G:1.3125 L1:0.1644 | 11.7s
Epoch   12/100 | D:0.5775 G:1.2205 L1:0.1615 | 6.4s
Epoch   13/100 | D:0.5602 G:1.2545 L1:0.1613 | 6.0s
Epoch   14/100 | D:0.5504 G:1.2067 L1:0.1542 | 6.0s
Epoch   15/100 | D:0.5785 G:1.2310 L1:0.1532 | 6.1s
Epoch   16/100 | D:0.5697 G:1.1576 L1:0.1458 | 6.1s
Epoch   17/100 | D:0.5354 G:1.1989 L1:0.1429 | 6.2s
Epoch   18/100 | D:0.5998 G:1.1381 L1:0.144

In [ ]:
%cd /content/drive/MyDrive/vangogh_restore

!python step2_train_pix2pix.py \
  --data_dir ./dataset_restore \
  --output_dir ./output_pix2pix \
  --epochs 5 \
  --batch_size 2

[Errno 2] No such file or directory: '/content/drive/MyDrive/vangogh_restore'
/content
python3: can't open file '/content/step2_train_pix2pix.py': [Errno 2] No such file or directory


In [ ]:
!python step3_evaluate.py \
  --data_dir   ./dataset_restore/test \
  --model_path ./output_pix2pix/checkpoints/generator_best.pth \
  --output_dir ./eval_results

Device: cuda
Model loaded: ./output_pix2pix/checkpoints/generator_best.pth
Đánh giá 38 ảnh...

KẾT QUẢ ĐÁNH GIÁ TỔNG HỢP
      ssim_degraded  ssim_restored  ssim_delta  psnr_degraded  psnr_restored  psnr_delta  edge_iou_degraded  edge_iou_restored  edge_iou_delta
mean         0.6255         0.7530      0.1275        16.5405        19.9008      3.3613             0.4031             0.4297          0.0267
std          0.0945         0.0370      0.0683         2.1255         1.5722      2.3513             0.0715             0.0505          0.0306
min          0.4064         0.6868      0.0091        12.9700        16.3700     -3.6800             0.2285             0.3172         -0.0151
max          0.7893         0.8209      0.3056        22.6700        22.7600      9.1000             0.5311             0.5260          0.0950

Phân loại chất lượng ảnh phục hồi:
quality
trung bình    33
tốt            5
fail           0

Kết quả lưu tại: ./eval_results/
  metrics.csv      — số liệu từng ả

In [ ]:
import os
from PIL import Image

test_pair_dir = "./dataset_restore/test"
input_dir = "./dataset_restore/test_inputs"
original_dir = "./dataset_restore/test_originals"

os.makedirs(input_dir, exist_ok=True)
os.makedirs(original_dir, exist_ok=True)

for fname in os.listdir(test_pair_dir):
    if not fname.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    path = os.path.join(test_pair_dir, fname)
    img = Image.open(path).convert("RGB")
    w, h = img.size
    half = w // 2

    degraded = img.crop((0, 0, half, h))
    original = img.crop((half, 0, w, h))

    base = fname.replace("_pair", "")
    degraded.save(os.path.join(input_dir, base))
    original.save(os.path.join(original_dir, base))

print("Đã tách degraded và original test.")

Đã tách degraded và original test.


In [ ]:
!ls dataset_restore/test_inputs | head
!ls dataset_restore/test_originals | head
!ls eval_results/restored | head

A L Arlesienne Madame Ginoux with Gloves and Umbre.jpg
Almond Tree in Blossom.jpg
A Pork-Butcher s Shop Seen from a Window.jpg
Apricot Trees in Blossom 2.jpg
Farmhouse in a Wheat Field.jpg
Farmhouse in Provence.jpg
Flowering Garden with Path.jpg
Garden Behind a House.jpg
Interior of a Restaurant in Arles.jpg
La Berceuse Augustine Roulin 3.jpg
A L Arlesienne Madame Ginoux with Gloves and Umbre.jpg
Almond Tree in Blossom.jpg
A Pork-Butcher s Shop Seen from a Window.jpg
Apricot Trees in Blossom 2.jpg
Farmhouse in a Wheat Field.jpg
Farmhouse in Provence.jpg
Flowering Garden with Path.jpg
Garden Behind a House.jpg
Interior of a Restaurant in Arles.jpg
La Berceuse Augustine Roulin 3.jpg
ls: cannot access 'eval_results/restored': No such file or directory


In [ ]:
!python A1_color_analysis.py \
  --original_dir ./dataset_restore/test_originals \
  --degraded_dir ./dataset_restore/test_inputs \
  --restored_dir ./eval_results/restored \
  --output_dir ./color_analysis


[1] Đang đọc ảnh...
  Đọc được 38 ảnh từ: ./dataset_restore/test_originals
  Đọc được 38 ảnh từ: ./dataset_restore/test_inputs
  Đọc được 0 ảnh từ: ./eval_results/restored

[2] Vẽ histogram HSV so sánh...
  Đã lưu: ./color_analysis/hsv_histograms.jpg

[3] Trích xuất color palette...
  Đang trích xuất palette màu (K-Means)...
  Đã lưu: ./color_analysis/color_palette.jpg

[4] Tính số liệu chi tiết từng ảnh...

[5] Tính độ tương đồng histogram (Bhattacharyya)...
PHÂN TÍCH MÀU SẮC ĐẶC TRƯNG VAN GOGH

Thống kê trung bình theo nhóm:

  [ORIGINAL]
  Saturation trung bình : 122.8/255
  Tỉ lệ pixel vàng      : 23.67%
  Tỉ lệ pixel xanh dương: 3.07%
  Tỉ lệ màu bão hòa cao : 34.42%

  [DEGRADED]
  Saturation trung bình : 22.8/255
  Tỉ lệ pixel vàng      : 0.33%
  Tỉ lệ pixel xanh dương: 1.17%
  Tỉ lệ màu bão hòa cao : 1.51%


Top 5 màu đặc trưng nhất (K-Means):
  #4C4831  (RGB: 76,72,49) — 16.4% pixel
  #6A795A  (RGB: 106,121,90) — 14.9% pixel
  #B28947  (RGB: 178,137,71) — 14.2% pixel
  #DCB45

In [ ]:
!python A2_texture_analysis.py \
  --original_dir ./dataset_restore/test_originals \
  --degraded_dir ./dataset_restore/test_inputs \
  --restored_dir ./eval_results/restored \
  --output_dir ./texture_analysis


[1] Đang đọc ảnh (grayscale)...
  Đọc được 38 ảnh từ: ./dataset_restore/test_originals
  Đọc được 38 ảnh từ: ./dataset_restore/test_inputs
  Đọc được 0 ảnh từ: ./eval_results/restored

[2] Tính Gabor energy theo hướng...
  Đã lưu: ./texture_analysis/gabor_orientation.jpg

[3] Vẽ heatmap Gabor trên ảnh mẫu...
  Đã lưu: ./texture_analysis/gabor_heatmap_sample.jpg

[4] Tính LBP histogram...
  Đã lưu: ./texture_analysis/lbp_histogram.jpg

[5] Tính số liệu texture chi tiết từng ảnh...

KẾT QUẢ PHÂN TÍCH TEXTURE

  [ORIGINAL]
  gabor_total_energy          : 3.61526 ± 0.55414
  lbp_entropy                 : 2.84988 ± 0.21163
  gradient_mean               : 0.44598 ± 0.10722

  [DEGRADED]
  gabor_total_energy          : 3.65274 ± 0.53474
  lbp_entropy                 : 2.28010 ± 0.10492
  gradient_mean               : 0.55188 ± 0.08524

Hoàn thành! Kết quả tại: ./texture_analysis/


In [ ]:
!python A3_style_report.py \
  --color_dir ./color_analysis \
  --texture_dir ./texture_analysis \
  --output_dir ./style_report


Bảng Style Score:
   group  style_score_overall
degraded                0.474
original                1.000

Báo cáo tổng hợp lưu tại: ./style_report/vangogh_style_report.jpg
→ Chèn file này trực tiếp vào Word/báo cáo NCKH.


In [11]:
%cd /content/drive/MyDrive/vangogh_restore

/content/drive/MyDrive/vangogh_restore


In [12]:
!python -c "import ast; ast.parse(open('step1_patch_blur.py').read()); print('OK')"

OK


In [13]:
!python step1_patch_blur.py \
  --preview "./vangogh_color/Almond Tree in Blossom.jpg"

Preview lưu tại: degradation_preview.jpg
→ Mở file này để chọn loại suy giảm muốn dùng trước khi chạy dataset.


In [14]:
!rm -rf dataset_blur output_blur eval_blur

!python step1_patch_blur.py \
  --input_dir ./vangogh_color \
  --output_dir ./dataset_blur \
  --deg_type blur_only \
  --train_ratio 0.8

Tìm thấy 188 ảnh. Deg type: [blur_only]
  Train: 150 | Test: 38
Xong! 188 cặp ảnh → ./dataset_blur/


In [15]:
!python step2_train_pix2pix.py \
  --data_dir ./dataset_blur \
  --output_dir ./output_blur \
  --epochs 5 \
  --batch_size 2

Device: cuda
Train: 150 ảnh | Test: 38 ảnh

Bắt đầu train 5 epochs...
Epoch    1/5 | D:0.7517 G:0.8434 L1:0.2346 | 7.8s
Epoch    2/5 | D:0.6504 G:0.9312 L1:0.1754 | 6.9s
Epoch    3/5 | D:0.4579 G:1.5721 L1:0.1569 | 7.1s
Epoch    4/5 | D:0.4416 G:1.8483 L1:0.1480 | 7.0s
Epoch    5/5 | D:0.4821 G:1.7121 L1:0.1414 | 7.2s

Train xong! Checkpoint tốt nhất lưu tại: ./output_blur/checkpoints/generator_best.pth
Ảnh mẫu lưu tại: ./output_blur/samples/


In [16]:
!python step2_train_pix2pix.py \
  --data_dir ./dataset_blur \
  --output_dir ./output_blur \
  --epochs 100 \
  --batch_size 4

Device: cuda
Train: 150 ảnh | Test: 38 ảnh

Bắt đầu train 100 epochs...
Epoch    1/100 | D:0.7647 G:0.8976 L1:0.2531 | 6.6s
Epoch    2/100 | D:0.6919 G:0.8367 L1:0.1782 | 6.3s
Epoch    3/100 | D:0.6401 G:0.9803 L1:0.1615 | 6.2s
Epoch    4/100 | D:0.4580 G:1.4471 L1:0.1503 | 6.3s
Epoch    5/100 | D:0.3912 G:1.7790 L1:0.1508 | 6.3s
Epoch    6/100 | D:0.4673 G:1.6155 L1:0.1509 | 6.4s
Epoch    7/100 | D:0.4602 G:1.6555 L1:0.1493 | 6.6s
Epoch    8/100 | D:0.4692 G:1.6645 L1:0.1465 | 6.6s
Epoch    9/100 | D:0.4811 G:1.5394 L1:0.1370 | 6.8s
Epoch   10/100 | D:0.4807 G:1.5316 L1:0.1355 | 6.8s
Epoch   11/100 | D:0.4802 G:1.5797 L1:0.1342 | 10.7s
Epoch   12/100 | D:0.5013 G:1.5445 L1:0.1347 | 7.1s
Epoch   13/100 | D:0.5040 G:1.4976 L1:0.1299 | 6.5s
Epoch   14/100 | D:0.5110 G:1.3963 L1:0.1289 | 6.6s
Epoch   15/100 | D:0.5247 G:1.3989 L1:0.1288 | 6.6s
Epoch   16/100 | D:0.5498 G:1.3079 L1:0.1269 | 6.6s
Epoch   17/100 | D:0.5532 G:1.3055 L1:0.1258 | 6.6s
Epoch   18/100 | D:0.5631 G:1.2784 L1:0.123

In [17]:
!python step3_evaluate.py \
  --data_dir ./dataset_blur/test \
  --model_path ./output_blur/checkpoints/generator_best.pth \
  --output_dir ./eval_blur

Device: cuda
Model loaded: ./output_blur/checkpoints/generator_best.pth
Đánh giá 38 ảnh...

KẾT QUẢ ĐÁNH GIÁ TỔNG HỢP
      ssim_degraded  ssim_restored  ssim_delta  psnr_degraded  psnr_restored  psnr_delta  edge_iou_degraded  edge_iou_restored  edge_iou_delta
mean         0.4064         0.5395      0.1331        20.9850        21.9803      0.9955             0.0380             0.1998          0.1618
std          0.0962         0.0681      0.0395         1.8311         1.9373      0.4646             0.0205             0.0150          0.0238
min          0.2575         0.4284      0.0533        17.0000        17.7300      0.3600             0.0028             0.1656          0.1064
max          0.6184         0.6716      0.1992        25.0300        26.8900      2.2300             0.0886             0.2278          0.2045

Phân loại chất lượng ảnh phục hồi:
quality
fail          30
trung bình     8
tốt            0

Kết quả lưu tại: ./eval_blur/
  metrics.csv      — số liệu từng ảnh
  s

In [18]:
!python step1_patch_blur.py \
  --input_dir ./vangogh_color \
  --output_dir ./dataset_blur_noise \
  --deg_type blur_noise

Tìm thấy 188 ảnh. Deg type: [blur_noise]
  Train: 150 | Test: 38
Xong! 188 cặp ảnh → ./dataset_blur_noise/


In [19]:
!python step2_train_pix2pix.py \
  --data_dir ./dataset_blur_noise \
  --output_dir ./output_blur_noise \
  --epochs 100

Device: cuda
Train: 150 ảnh | Test: 38 ảnh

Bắt đầu train 100 epochs...
Epoch    1/100 | D:0.7851 G:0.8428 L1:0.2415 | 6.9s
Epoch    2/100 | D:0.7002 G:0.7568 L1:0.1645 | 5.9s
Epoch    3/100 | D:0.6849 G:0.7782 L1:0.1561 | 6.0s
Epoch    4/100 | D:0.6665 G:0.8733 L1:0.1436 | 5.9s
Epoch    5/100 | D:0.5650 G:1.1087 L1:0.1408 | 6.0s
Epoch    6/100 | D:0.4337 G:1.5573 L1:0.1343 | 6.0s
Epoch    7/100 | D:0.4877 G:1.5119 L1:0.1391 | 6.1s
Epoch    8/100 | D:0.4970 G:1.5245 L1:0.1377 | 6.3s
Epoch    9/100 | D:0.5051 G:1.4911 L1:0.1355 | 6.3s
Epoch   10/100 | D:0.4768 G:1.5455 L1:0.1348 | 6.4s
Epoch   11/100 | D:0.5346 G:1.4662 L1:0.1344 | 7.8s
Epoch   12/100 | D:0.5231 G:1.4123 L1:0.1332 | 6.9s
Epoch   13/100 | D:0.5302 G:1.4777 L1:0.1319 | 6.6s
Epoch   14/100 | D:0.5229 G:1.3725 L1:0.1327 | 6.6s
Epoch   15/100 | D:0.5314 G:1.2851 L1:0.1308 | 6.7s
Epoch   16/100 | D:0.5198 G:1.3085 L1:0.1317 | 6.8s
Epoch   17/100 | D:0.5494 G:1.3502 L1:0.1312 | 6.8s
Epoch   18/100 | D:0.5466 G:1.2879 L1:0.1297

In [20]:
!python step3_evaluate.py \
  --data_dir ./dataset_blur_noise/test \
  --model_path ./output_blur_noise/checkpoints/generator_best.pth \
  --output_dir ./eval_blur_noise

Device: cuda
Model loaded: ./output_blur_noise/checkpoints/generator_best.pth
Đánh giá 38 ảnh...

KẾT QUẢ ĐÁNH GIÁ TỔNG HỢP
      ssim_degraded  ssim_restored  ssim_delta  psnr_degraded  psnr_restored  psnr_delta  edge_iou_degraded  edge_iou_restored  edge_iou_delta
mean         0.3056         0.4110      0.1053        19.9482        20.8168      0.8684             0.2004             0.1921         -0.0083
std          0.0394         0.0624      0.0323         1.2940         1.6472      0.4001             0.0223             0.0154          0.0133
min          0.2356         0.3022      0.0624        16.7800        17.0700      0.2900             0.1428             0.1575         -0.0393
max          0.3923         0.5355      0.2080        22.6400        25.0200      2.3800             0.2300             0.2150          0.0155

Phân loại chất lượng ảnh phục hồi:
quality
fail          38
trung bình     0
tốt            0

Kết quả lưu tại: ./eval_blur_noise/
  metrics.csv      — số liệu 

In [21]:
!python step1_patch_blur.py \
  --input_dir ./vangogh_color \
  --output_dir ./dataset_gray_blur_noise \
  --deg_type gray_blur_noise

Tìm thấy 188 ảnh. Deg type: [gray_blur_noise]
  Train: 150 | Test: 38
Xong! 188 cặp ảnh → ./dataset_gray_blur_noise/


In [22]:
!python step2_train_pix2pix.py \
  --data_dir ./dataset_gray_blur_noise \
  --output_dir ./output_gray_blur_noise \
  --epochs 100

Device: cuda
Train: 150 ảnh | Test: 38 ảnh

Bắt đầu train 100 epochs...
Epoch    1/100 | D:0.7410 G:0.9214 L1:0.2723 | 6.6s
Epoch    2/100 | D:0.5422 G:1.1730 L1:0.2213 | 5.9s
Epoch    3/100 | D:0.4565 G:1.5862 L1:0.2203 | 5.8s
Epoch    4/100 | D:0.4542 G:1.7140 L1:0.2175 | 5.9s
Epoch    5/100 | D:0.4682 G:1.6294 L1:0.2138 | 5.9s
Epoch    6/100 | D:0.4884 G:1.4910 L1:0.2113 | 6.0s
Epoch    7/100 | D:0.5590 G:1.4883 L1:0.2058 | 6.1s
Epoch    8/100 | D:0.5482 G:1.3967 L1:0.2031 | 6.1s
Epoch    9/100 | D:0.5123 G:1.4548 L1:0.2040 | 6.3s
Epoch   10/100 | D:0.4894 G:1.4663 L1:0.1965 | 6.2s
Epoch   11/100 | D:0.4342 G:1.5846 L1:0.1935 | 6.4s
Epoch   12/100 | D:0.4940 G:1.5800 L1:0.1853 | 6.4s
Epoch   13/100 | D:0.4649 G:1.5588 L1:0.1895 | 6.5s
Epoch   14/100 | D:0.4491 G:1.5793 L1:0.1861 | 6.5s
Epoch   15/100 | D:0.4730 G:1.6260 L1:0.1861 | 6.6s
Epoch   16/100 | D:0.5092 G:1.5244 L1:0.1808 | 6.7s
Epoch   17/100 | D:0.5219 G:1.4130 L1:0.1830 | 6.7s
Epoch   18/100 | D:0.4944 G:1.5183 L1:0.1779

In [23]:
!python step3_evaluate.py \
  --data_dir ./dataset_gray_blur_noise/test \
  --model_path ./output_gray_blur_noise/checkpoints/generator_best.pth \
  --output_dir ./eval_gray_blur_noise

Device: cuda
Model loaded: ./output_gray_blur_noise/checkpoints/generator_best.pth
Đánh giá 38 ảnh...

KẾT QUẢ ĐÁNH GIÁ TỔNG HỢP
      ssim_degraded  ssim_restored  ssim_delta  psnr_degraded  psnr_restored  psnr_delta  edge_iou_degraded  edge_iou_restored  edge_iou_delta
mean         0.2725         0.3176      0.0451        15.4539        16.9479      1.4950             0.2005             0.2054          0.0049
std          0.0368         0.0510      0.0300         1.5166         1.1155      1.9405             0.0223             0.0207          0.0057
min          0.2085         0.2373     -0.0093        12.9100        14.5800     -3.1100             0.1424             0.1467         -0.0084
max          0.3633         0.4432      0.1601        18.9300        20.2700      7.3600             0.2309             0.2346          0.0186

Phân loại chất lượng ảnh phục hồi:
quality
fail          38
trung bình     0
tốt            0

Kết quả lưu tại: ./eval_gray_blur_noise/
  metrics.csv      